# Lakehouse
Um Lakehouse é uma novo conceito que permite aos usuários fazer tudo, desde BI, análise SQL, Machine Learnig em uma única plataforma.
O Lakehouse tem uma abordagem opinativa para construir Data Lakes, adicionando atributos de Data Warehouses - confiabilidade, desempenho e qualidade, mantendo a abertura e escala de Data Lakes.

- Referências
  - Artigo: Gerenciamento de Dados: dos Dados ao Lakehouse ([link](https://blog.compassouol.com/tech/gerenciamento-de-dados-dos-dados-ao-lakehouse/))
  - Camadas Delta Lake ([link](https://live-delta-io.pantheonsite.io/wp-content/uploads/2019/04/Delta-Lake-marketecture-0423c.png))
  - Acessando o Databricks pelo Power BI ([link](https://blogdozouza.wordpress.com/2021/03/21/lendo-tabelas-do-databricks-community-pelo-power-bi/))

## Criando um Lakehouse com o Delta Lake (Databricks)
- Ferramenta - [*Databricks Free Edition*](https://login.databricks.com/signup?provider=DB_FREE_TIER)
- Usaremos *Spark SQL* e ***Delta Lake*** para ingerir dados e criar as tabelas do ERP para serem nossa única fonte de verdade (*Single Source of Truth - SSOT*). 
- A partir daí, usaremos as tabelas *Silver* para construir nosso ***Data Mart*** de Vendas.
- Também usaremos ***Delta Lake*** para realizar manutenções nas tabelas (inserts, updates e deletes) e tornar nossos dados mais em "tempo real".
- Ler pelo Power BI

### Criando o banco de dados (chamaremos de: Lake)
- que usaremos para análises de BI

In [0]:
-- Já cria o banco de dados dentro do Workspace
CREATE DATABASE IF NOT EXISTS lake;
USE lake;

Salve os [arquivos](https://github.com/aasouzaconsult/lakehouse/tree/main/Ambiente%20demonstra%C3%A7%C3%A3o/arquivos) no volume que criará

In [0]:
%fs
ls /Volumes/workspace/lake/lake

path,name,size,modificationTime
dbfs:/Volumes/workspace/lake/lake/tbdependente.csv,tbdependente.csv,257,1791579067000
dbfs:/Volumes/workspace/lake/lake/tbdependente_20210607_0850.csv,tbdependente_20210607_0850.csv,79,1791579068000
dbfs:/Volumes/workspace/lake/lake/tbvendas.csv,tbvendas.csv,1015,1791579067000
dbfs:/Volumes/workspace/lake/lake/tbvendas_20210607.csv,tbvendas_20210607.csv,929,1791579068000
dbfs:/Volumes/workspace/lake/lake/tbvendedor.csv,tbvendedor.csv,160,1791579067000


### Criando a camada Bronze (RAW) - Tabelas de Ingestão
- criando as tabelas de ingestão, conforme arquivos recebidos do ERP (contidos no nosso Lake)

In [0]:
-- Aqui pegaremos informações históricas de 3 tabelas do ERP
DROP TABLE IF EXISTS tbdependente_hist;
DROP TABLE IF EXISTS tbvendas_hist;
DROP TABLE IF EXISTS tbvendedor_hist;

-- Carga Batch
-- Uma carga com dados históricos de dependentes
CREATE TABLE tbdependente_hist AS
SELECT * FROM read_files('/Volumes/workspace/lake/lake/tbdependente.csv', format => 'csv', header => true, sep => ';');

-- Uma carga com dados históricos de vendas
CREATE TABLE tbvendas_hist AS
SELECT * FROM read_files('/Volumes/workspace/lake/lake/tbvendas.csv', format => 'csv', header => true, sep => ';');

-- Uma carga com dados históricos de vendedores
CREATE TABLE tbvendedor_hist AS
SELECT * FROM read_files('/Volumes/workspace/lake/lake/tbvendedor.csv', format => 'csv', header => true, sep => ';');

-- Carga Stream
-- DMS da Amazon - pode pegar do SQL Server (via CDC) e levar ao S3 em formato .parquet e essa entrada ser constante... Onde entraria um job para fazer o upinsert no Delta Lake por exemplo

num_affected_rows,num_inserted_rows


#### Visualizando as tabelas da camanda Bronze
- lendo estas tabelas utilizando SQL :)

In [0]:
-- Visualizando a carga de dependentes
select * from tbdependente_hist;

-- vale lembrar:
-- - nomenclatura dos campos
-- - padronização (campo sexo)

CdDep,NmDep,DtNasc,SxDep,MatFunc,delete,_rescued_data
1,Dependente 1,2010-02-02,Masc,1,0,null
2,Dependente 2,2012-05-03,Masc,3,0,null
3,Dependente 3,2014-05-14,Fem,3,0,null
4,Dependente 4,2016-03-05,Fem,4,0,null
5,Dependente 5,2017-07-06,Masc,4,0,null
6,Dependente 6,2018-04-07,Fem,9,0,null


In [0]:
-- Visualizando a carga de vendas
select * from tbvendas_hist;

CdVen,DtVen,CdCli,NmCli,Cidade,Estado,Pais,CdPro,NmPro,TpPro,Qtd,Und,VrUnt,Total,CdVdd,delete,_rescued_data
1,2021-01-20,1,Cliente AA,Florianopolis,Santa Catarina,Brasil,1,Produto A,A,4000,KG,8,32000,2,0,null
2,2021-05-31,2,Cliente AB,Fortaleza,Ceara,Brasil,2,Produto B,A,730,KG,20,14600,5,0,null
3,2021-06-01,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,4200,KG,10,42000,4,0,null
4,2021-06-02,3,Cliente BC,Baturite,Ceara,Brasil,1,Produto A,A,250,KG,10,2500,2,0,null
5,2021-06-03,4,Cliente CC,Fortaleza,Ceara,Brasil,1,Produto A,A,4500,KG,10,45000,1,0,null
6,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,11200,KG,10,112000,3,0,null
7,2021-06-04,6,Cliente DD,Goiania,Goias,Brasil,1,Produto A,A,12500,KG,10,125000,2,0,null
8,2021-06-04,7,Cliente DE,Joao Pessoa,Paraiba,Brasil,1,Produto A,A,12000,KG,10,120000,2,0,null
9,2021-06-04,8,Cliente EE,Natal,Rio Grande do Norte,Brasil,1,Produto A,A,17500,KG,10,175000,3,0,null
10,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,24000,KG,10,240000,1,0,null


In [0]:
-- Visualizando a carga de vendedor
select * from tbvendedor_hist;

CdVdd,NmVdd,SxVdd,PercComissao,MatFunc,delete,_rescued_data
1,Vendedor 1,1,1,1,0,null
2,Vendedor 2,1,1,2,0,null
3,Vendedor 3,1,1,3,0,null
4,Vendedor 4,1,2,4,0,null
5,Vendedor 5,0,1,5,0,null


### Criando a camada Silver (trusted) - Tratamento de dados

**Criando a tabela silver de Dependentes**
- salvando em formato colunar e otimizado (DELTA )
- semanticamente amigável
- padronização de campos (*data quality*)

In [0]:
DROP TABLE IF EXISTS silver_dependente;   
                                                          
CREATE TABLE workspace.lake.silver_dependente
USING DELTA
PARTITIONED BY (SexoDependente)               -- column used to partition the data
AS (                                                      
  SELECT CdDep   as CodigoDependente -- Semanticamente amigaveis
       , NmDep   as NomeDependente
       , DtNasc  as DataNascimento
       , case when SxDep = "Masc" then 1 else 0 end as SexoDependente -- padronizacao
       , MatFunc as MatriculaFuncionario
       , delete -- 0 ativo e 1 apagado
    FROM tbdependente_hist 
)

num_affected_rows,num_inserted_rows


**Criando a tabela silver de Vendas**
- salvando em formato colunar e otimizado (DELTA)
- semanticamente amigável
- conversões de campos numéricos

In [0]:
DROP TABLE IF EXISTS silver_vendas;   
                                                          
CREATE TABLE workspace.lake.silver_vendas
USING DELTA
PARTITIONED BY (Estado)
AS
SELECT
    CdVen                          AS CodigoVenda,
    DtVen                          AS DataVenda,
    CdCli                          AS CodigoCliente,
    NmCli                          AS NomeCliente,
    Cidade,
    Estado,
    Pais,
    CdPro                          AS CodigoProduto,
    NmPro                          AS NomeProduto,
    TpPro                          AS TipoProduto,
    CAST(Qtd AS INT)               AS Quantidade,
    Und,
    CAST(VrUnt AS DECIMAL(18, 2))  AS ValorUnitario,
    CdVdd                          AS CodigoVendedor,
    `delete`                       AS RegistroExcluido
FROM workspace.lake.tbvendas_hist;

num_affected_rows,num_inserted_rows


**Criando a tabela silver de Vendedor**
- salvando em formato colunar e otimizado (DELAT )
- semanticamente amigável
- padronização de campos (*data quality*)
- conversões de campos numéricos

In [0]:
DROP TABLE IF EXISTS silver_vendedor;   
                                                          
CREATE TABLE workspace.lake.silver_vendedor
USING DELTA
PARTITIONED BY (SexoVendedor)
AS
 (                                                      
  SELECT CdVdd as CodigoVendedor
       , NmVdd as NomeVendedor
       , SxVdd as SexoVendedor
       , cast(PercComissao as decimal(18,2)) as PercentualComissao
       , MatFunc as MatriculaFuncionario
       , delete
    FROM tbvendedor_hist
)

num_affected_rows,num_inserted_rows


#### Detalhando a tabela de Dependentes da Camada Silver (silver_dependente)
Consulte também as demais tabelas

In [0]:
DESCRIBE DETAIL silver_dependente;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,3bf5cb27-e12a-49e2-905d-53aa7e7656f3,workspace.lake.silver_dependente,null,,2026-10-09T21:36:06.646Z,2026-10-09T21:36:09.000Z,List(SexoDependente),List(),2,4244,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> ad99a0d5-e9f9-467c-9c44-0deea58ac952)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
SELECT
    table_catalog,
    table_schema,
    table_name,
    table_type,
    data_source_format,
    storage_path
FROM workspace.information_schema.tables
WHERE table_schema = 'lake'
  AND table_name = 'silver_dependente';

table_catalog,table_schema,table_name,table_type,data_source_format,storage_path
workspace,lake,silver_dependente,MANAGED,DELTA,


#### Consultando as tabelas silver

- aqui a tabela já dá suporte aos recursos do Lakehouse

In [0]:
-- Consultando em formato Delta (dependente)
select * from silver_dependente;

CodigoDependente,NomeDependente,DataNascimento,SexoDependente,MatriculaFuncionario,delete
1,Dependente 1,2010-02-02,1,1,0
2,Dependente 2,2012-05-03,1,3,0
5,Dependente 5,2017-07-06,1,4,0
3,Dependente 3,2014-05-14,0,3,0
4,Dependente 4,2016-03-05,0,4,0
6,Dependente 6,2018-04-07,0,9,0


In [0]:
-- Consultando em formato Delta (Vendas)
select * from silver_vendas;

CodigoVenda,DataVenda,CodigoCliente,NomeCliente,Cidade,Estado,Pais,CodigoProduto,NomeProduto,TipoProduto,Quantidade,Und,ValorUnitario,CodigoVendedor,RegistroExcluido
3,2021-06-01,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,4200,KG,10.00,4,0
6,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,11200,KG,10.00,3,0
10,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,24000,KG,10.00,1,0
2,2021-05-31,2,Cliente AB,Fortaleza,Ceara,Brasil,2,Produto B,A,730,KG,20.00,5,0
4,2021-06-02,3,Cliente BC,Baturite,Ceara,Brasil,1,Produto A,A,250,KG,10.00,2,0
5,2021-06-03,4,Cliente CC,Fortaleza,Ceara,Brasil,1,Produto A,A,4500,KG,10.00,1,0
1,2021-01-20,1,Cliente AA,Florianopolis,Santa Catarina,Brasil,1,Produto A,A,4000,KG,8.00,2,0
9,2021-06-04,8,Cliente EE,Natal,Rio Grande do Norte,Brasil,1,Produto A,A,17500,KG,10.00,3,0
8,2021-06-04,7,Cliente DE,Joao Pessoa,Paraiba,Brasil,1,Produto A,A,12000,KG,10.00,2,0
7,2021-06-04,6,Cliente DD,Goiania,Goias,Brasil,1,Produto A,A,12500,KG,10.00,2,0


In [0]:
-- Consultando em formato Delta (vendedor)
select * from silver_vendedor;

CodigoVendedor,NomeVendedor,SexoVendedor,PercentualComissao,MatriculaFuncionario,delete
1,Vendedor 1,1,1.00,1,0
2,Vendedor 2,1,1.00,2,0
3,Vendedor 3,1,1.00,3,0
4,Vendedor 4,1,2.00,4,0
5,Vendedor 5,0,1.00,5,0


### Criando a camada Gold (refined) - Análise
- A seguir, criaremos uma tabela Delta - Gold. Faremos isso criando uma tabela agregada a partir dos dados na tabela delta: vendas_silver que acabamos de criar. 

#### Criando uma tabela de agregação
- Valor total de vendas por Estado

In [0]:
----------------------------
-- Agregação Total vendas --
----------------------------

DROP TABLE IF EXISTS workspace.lake.gold_totalvendas;

CREATE TABLE workspace.lake.gold_totalvendas
USING DELTA
AS
(
    SELECT Estado,
           CAST(
               SUM(Quantidade * ValorUnitario)
               AS DECIMAL(18,2)
           ) AS ValorTotal
    FROM workspace.lake.silver_vendas
    WHERE RegistroExcluido = 0
    GROUP BY Estado
);

num_affected_rows,num_inserted_rows


In [0]:
-- Visualizando em tabelas
select * from gold_totalvendas;

-- Visualizando em barras

Estado,ValorTotal
Minas Gerais,394000.00
Ceara,62100.00
Santa Catarina,32000.00
Rio Grande do Norte,175000.00
Paraiba,120000.00
Goias,125000.00


Databricks visualization. Run in Databricks to view.

### Criando um DW aqui no Lakehouse :)

- Das tabelas Silver (DELTA), já podemos montar nosso ***Data Marts*** de Vendas, salvando na camada superior gold ou criando *views* (no case aqui, criamos *views*)

##### Criando as Dimensões e Fatos

In [0]:
-------------------------
-- Dimensão Dependente --
-------------------------
CREATE OR REPLACE VIEW gold_dim_dependente
AS (
  SELECT *
       , '11000082' as INEP_ESCOLA -- todos os filhos dos vendedores estudam na mesma escola (benefício)
    FROM silver_dependente
   WHERE delete = 0
)

In [0]:
-- Consultando a view dependente
SELECT * FROM gold_dim_dependente;

CodigoDependente,NomeDependente,DataNascimento,SexoDependente,MatriculaFuncionario,delete,INEP_ESCOLA
1,Dependente 1,2010-02-02,1,1,0,11000082
2,Dependente 2,2012-05-03,1,3,0,11000082
5,Dependente 5,2017-07-06,1,4,0,11000082
3,Dependente 3,2014-05-14,0,3,0,11000082
4,Dependente 4,2016-03-05,0,4,0,11000082
6,Dependente 6,2018-04-07,0,9,0,11000082


In [0]:
-------------------------
-- Dimensão Vendedores --
-------------------------
CREATE OR REPLACE VIEW gold_dim_vendedor
AS (
  SELECT *
    FROM silver_vendedor
   WHERE delete = 0
)

In [0]:
-- Consultando a view vendedor
SELECT * FROM gold_dim_vendedor;

CodigoVendedor,NomeVendedor,SexoVendedor,PercentualComissao,MatriculaFuncionario,delete
1,Vendedor 1,1,1.00,1,0
2,Vendedor 2,1,1.00,2,0
3,Vendedor 3,1,1.00,3,0
4,Vendedor 4,1,2.00,4,0
5,Vendedor 5,0,1.00,5,0


**Tabela de Vendas - Silver**
- Além do FATO Venda em si, esta tabela contem:
  - informações de clientes (que iremos transformar em uma *dimensão cliente*)
  - informações de produto (que iremos transformar em uma *dimensão produto*)

In [0]:
SELECT * FROM silver_vendas

CodigoVenda,DataVenda,CodigoCliente,NomeCliente,Cidade,Estado,Pais,CodigoProduto,NomeProduto,TipoProduto,Quantidade,Und,ValorUnitario,CodigoVendedor,RegistroExcluido
3,2021-06-01,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,4200,KG,10.00,4,0
6,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,11200,KG,10.00,3,0
10,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,24000,KG,10.00,1,0
2,2021-05-31,2,Cliente AB,Fortaleza,Ceara,Brasil,2,Produto B,A,730,KG,20.00,5,0
4,2021-06-02,3,Cliente BC,Baturite,Ceara,Brasil,1,Produto A,A,250,KG,10.00,2,0
5,2021-06-03,4,Cliente CC,Fortaleza,Ceara,Brasil,1,Produto A,A,4500,KG,10.00,1,0
1,2021-01-20,1,Cliente AA,Florianopolis,Santa Catarina,Brasil,1,Produto A,A,4000,KG,8.00,2,0
9,2021-06-04,8,Cliente EE,Natal,Rio Grande do Norte,Brasil,1,Produto A,A,17500,KG,10.00,3,0
8,2021-06-04,7,Cliente DE,Joao Pessoa,Paraiba,Brasil,1,Produto A,A,12000,KG,10.00,2,0
7,2021-06-04,6,Cliente DD,Goiania,Goias,Brasil,1,Produto A,A,12500,KG,10.00,2,0


In [0]:
-----------------
-- dim Cliente --
-----------------
CREATE OR REPLACE VIEW gold_dim_cliente
AS (
  SELECT Distinct 
         CodigoCliente
       , NomeCliente
       , Cidade
       , Estado
       , Pais
    FROM silver_vendas
   WHERE RegistroExcluido = 0
)

In [0]:
-----------------
-- dim Produto --
-----------------
CREATE OR REPLACE VIEW gold_dim_produto
AS (
  SELECT DISTINCT
         CodigoProduto
       , NomeProduto   
       , TipoProduto
       , Und
    FROM silver_vendas
   WHERE RegistroExcluido = 0
)

In [0]:
-----------------
-- Fato Vendas --
-----------------
CREATE OR REPLACE VIEW gold_fato_vendas
AS (
  SELECT CodigoVenda
       , DataVenda
       , Quantidade
       , ValorUnitario
       , CodigoCliente
       , CodigoProduto
       , CodigoVendedor
    FROM silver_vendas
   WHERE RegistroExcluido = 0 
)

<h4> Consultando algumas views do Data Mart</h4>

In [0]:
select * from gold_fato_vendas;

CodigoVenda,DataVenda,Quantidade,ValorUnitario,CodigoCliente,CodigoProduto,CodigoVendedor
3,2021-06-01,4200,10.00,5,1,4
6,2021-06-04,11200,10.00,5,1,3
10,2021-06-04,24000,10.00,5,1,1
2,2021-05-31,730,20.00,2,2,5
4,2021-06-02,250,10.00,3,1,2
5,2021-06-03,4500,10.00,4,1,1
1,2021-01-20,4000,8.00,1,1,2
9,2021-06-04,17500,10.00,8,1,3
8,2021-06-04,12000,10.00,7,1,2
7,2021-06-04,12500,10.00,6,1,2


In [0]:
select * from gold_dim_produto;

CodigoProduto,NomeProduto,TipoProduto,Und
1,Produto A,A,KG
2,Produto B,A,KG


In [0]:
select * from gold_dim_dependente;

CodigoDependente,NomeDependente,DataNascimento,SexoDependente,MatriculaFuncionario,delete,INEP_ESCOLA
1,Dependente 1,2010-02-02,1,1,0,11000082
2,Dependente 2,2012-05-03,1,3,0,11000082
5,Dependente 5,2017-07-06,1,4,0,11000082
3,Dependente 3,2014-05-14,0,3,0,11000082
4,Dependente 4,2016-03-05,0,4,0,11000082
6,Dependente 6,2018-04-07,0,9,0,11000082


<h4> Exemplo de relacionamento entre Tabelas</h4>
- Fato_Vendas com a Dim_Produto

In [0]:
select ven.*
     , pro.nomeproduto
  from gold_fato_vendas ven
  join gold_dim_produto pro on pro.codigoproduto = ven.codigoproduto

CodigoVenda,DataVenda,Quantidade,ValorUnitario,CodigoCliente,CodigoProduto,CodigoVendedor,nomeproduto
3,2021-06-01,4200,10.00,5,1,4,Produto A
6,2021-06-04,11200,10.00,5,1,3,Produto A
10,2021-06-04,24000,10.00,5,1,1,Produto A
2,2021-05-31,730,20.00,2,2,5,Produto B
4,2021-06-02,250,10.00,3,1,2,Produto A
5,2021-06-03,4500,10.00,4,1,1,Produto A
1,2021-01-20,4000,8.00,1,1,2,Produto A
9,2021-06-04,17500,10.00,8,1,3,Produto A
8,2021-06-04,12000,10.00,7,1,2,Produto A
7,2021-06-04,12500,10.00,6,1,2,Produto A


### Anexando novos registros e atualizações a uma tabela delta existente

#### Updates na Tabela de Dependentes (Upsert)
- chegou mais uma atualização ma tabela de dependentes...

In [0]:
DROP TABLE IF EXISTS tbdependente_20210607_0850;

-- Carga Batch
-- Uma carga incremental com deletes

CREATE TABLE tbdependente_20210607_0850 AS
SELECT * FROM read_files('/Volumes/workspace/lake/lake/tbdependente_20210607_0850.csv', format => 'csv', header => true, sep => ';');

num_affected_rows,num_inserted_rows


<h4> Preparar view de updates </h4>
- utilizaremos posteriormente no *upsert*

In [0]:
-- Vamos escrever esses valores em uma visão temporária chamada updates. Isso será usado posteriormente para fazer o upsert de valores em nossa tabela

CREATE OR REPLACE TEMPORARY VIEW updates 
AS (
    SELECT CdDep   as CodigoDependente
         , NmDep   as NomeDependente
         , DtNasc  as DataNascimento
         , MatFunc as MatriculaFuncionario
         , delete -- 0 ativo e 1 apagado
         , case when SxDep = "Masc" then 1 else 0 end as SexoDependente
      FROM tbdependente_20210607_0850
)

<h4> Verificando os schemas das tabelas </h4>
- tanto a de dependente_silver (Atual)
- tanto da view de updates que criamos (pois iremos realizar o merge e tem que ter o mesmo schema - um dos segredos do conceito Lakehouse)

In [0]:
-- view the schema of the dependente_silver table.
DESCRIBE silver_dependente;

col_name,data_type,comment
CodigoDependente,int,null
NomeDependente,string,null
DataNascimento,date,null
SexoDependente,int,null
MatriculaFuncionario,int,null
delete,int,null
# Partition Information,,
# col_name,data_type,comment
SexoDependente,int,null


In [0]:
-- Descreva as atualizações
-- Usamos o comando DESCRIBE para visualizar o esquema da visualização de atualizações.
DESCRIBE updates

col_name,data_type,comment
CodigoDependente,int,null
NomeDependente,string,null
DataNascimento,string,null
MatriculaFuncionario,int,null
delete,int,null
SexoDependente,int,null


<h4> O que será atualizado? </h4>
- deletando o CodigoDependente 6

In [0]:
SELECT * FROM updates

CodigoDependente,NomeDependente,DataNascimento,MatriculaFuncionario,delete,SexoDependente
6,Dependente 6,07/04/2018,9,1,0


<h4> Preparar view de upsert </h4>
- preparamos a visualização de upserts que consiste em todos os registros nas visualizações de atualizações e inserções. Usamos o comando Spark SQL UNION para criar a visualização.
  - Neste momento, só teremos updates

In [0]:
CREATE OR REPLACE TEMPORARY VIEW upserts
AS (
    SELECT * FROM updates 
  --UNION ALL 
  --SELECT * FROM inserts
    )

<h4> Executando o Upsert </h4>
- Você pode fazer o upsert de dados em uma tabela Delta usando a operação de mesclagem. 
  - Esta operação é semelhante ao comando SQL MERGE, mas adicionou suporte para exclusões e outras condições em atualizações, inserções e exclusões.
  - Em outras palavras, o uso do comando Spark SQL MERGE fornece suporte total para uma operação de upsert.

In [0]:
MERGE INTO silver_dependente AS destino               -- Tabela que será atualizada ou receberá novos registros
USING upserts AS origem                                -- Dados novos ou alterados que serão processados

ON destino.CodigoDependente = origem.CodigoDependente -- Compara os registros pelo código do dependente

WHEN MATCHED THEN                                      -- Quando o dependente já existe na tabela
  UPDATE SET
    destino.delete = origem.delete                     -- Atualiza somente o campo delete, conforme a regra de negócio

WHEN NOT MATCHED THEN                                  -- Quando o dependente ainda não existe na tabela
  INSERT (
    CodigoDependente,
    NomeDependente,
    DataNascimento,
    MatriculaFuncionario,
    delete,
    SexoDependente
  )
  VALUES (
    origem.CodigoDependente,
    origem.NomeDependente,
    origem.DataNascimento,
    origem.MatriculaFuncionario,
    origem.delete,
    origem.SexoDependente
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


<h4> Viagem no Tempo... </h4>
- Versão 0

In [0]:
SELECT * FROM silver_dependente VERSION AS OF 0

CodigoDependente,NomeDependente,DataNascimento,SexoDependente,MatriculaFuncionario,delete
1,Dependente 1,2010-02-02,1,1,0
2,Dependente 2,2012-05-03,1,3,0
5,Dependente 5,2017-07-06,1,4,0
3,Dependente 3,2014-05-14,0,3,0
4,Dependente 4,2016-03-05,0,4,0
6,Dependente 6,2018-04-07,0,9,0


<h4> Viagem no Tempo... </h4>
- Versão recente...

In [0]:
-- Quando consultamos a tabela sem especificar uma versão, ela mostra a versão mais recente da tabela (dependente 6 esta como deletado)
SELECT * FROM silver_dependente 

CodigoDependente,NomeDependente,DataNascimento,SexoDependente,MatriculaFuncionario,delete
6,Dependente 6,2018-04-07,0,9,1
3,Dependente 3,2014-05-14,0,3,0
4,Dependente 4,2016-03-05,0,4,0
1,Dependente 1,2010-02-02,1,1,0
2,Dependente 2,2012-05-03,1,3,0
5,Dependente 5,2017-07-06,1,4,0


In [0]:
-- Automaticamente seu BI estará atualizado
select * from gold_dim_dependente;

CodigoDependente,NomeDependente,DataNascimento,SexoDependente,MatriculaFuncionario,delete,INEP_ESCOLA
3,Dependente 3,2014-05-14,0,3,0,11000082
4,Dependente 4,2016-03-05,0,4,0,11000082
1,Dependente 1,2010-02-02,1,1,0,11000082
2,Dependente 2,2012-05-03,1,3,0,11000082
5,Dependente 5,2017-07-06,1,4,0,11000082


<h4> Descrevendo a história da tabela dependente_silver... </h4>
- todas movimentações que fizemos...

In [0]:
-- O comando DESCRIBE HISTORY Spark SQL fornece informações de proveniência, incluindo a operação, usuário e assim por diante, para cada gravação em uma tabela. 
-- Observe que cada operação realizada na tabela recebe um número de versão. Esses são os números que usamos ao realizar uma consulta de viagem no tempo na tabela

DESCRIBE HISTORY silver_dependente;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-10-12T23:58:06.000Z,3982668590889822,aasouzaconsult@gmail.com,MERGE,"Map(predicate -> [""(CodigoDependente#16230 = CodigoDependente#16192)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(jobName -> null, jobOwnerId -> null, workspaceId -> 4360380715050102, jobId -> , jobRunId -> , runId -> , triggerType -> null)",null,01f1c698-c033-1edb-8ba9-e9be1b86d36b,null,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 2, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2216, numTargetBytesRemoved -> 2121, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5700, materializeSourceTimeMs -> 667, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2742, numTargetRowsUpdated -> 1, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2182)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
0,2026-10-09T21:36:09.000Z,3982668590889822,aasouzaconsult@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [""SexoDependente""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""ad99a0d5-e9f9-467c-9c44-0deea58ac952""}, statsOnLoad -> true)",null,Map(notebookId -> 1256435283671815),f3bc7943-ff87-4760-a7d7-d48947bd74d1,1009-210247-psrdxnx8-v2n,null,WriteSerializable,true,"Map(numFiles -> 2, numOutputRows -> 6, numOutputBytes -> 4244)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


#### Inserts, Updates, Deletes na Tabela de Vendas (Upsert)
- Aqui usaremos de forma didática as movimentações do dia da tabela vindo em um .csv
- Mas na vida real, por exemplo, poderá fazer replicação do ERP para o **Amazon S3** usando o *DMS (Data Migration Services)* via *CDC (Change Data Capture)* e montar algum jobs para realizar a leitura destes arquivos que chegam de forma automática.

In [0]:
-- Arquivo só com as alterações
DROP TABLE IF EXISTS tbvendas_20210607;

-- Carga Batch
-- Uma carga com dados d-1

CREATE TABLE tbvendas_20210607 AS
SELECT * FROM read_files('/Volumes/workspace/lake/lake/tbvendas_20210607.csv', format => 'csv', header => true, sep => ';');

num_affected_rows,num_inserted_rows


<h4> Criando a View de Upserts da Tabela de Vendas </h4>
- todas movimentações do dia anterior (no exemplo)...
- Criando view com os inserts, updates e deletes lógicos feitos no ERP (vendas) para adicionarmos na Silver (já com os dados tratados)

In [0]:
-- Criando view com os inserts, updates e deletes lógicos feitos no ERP (vendas) para adicionarmos na Silver (já com os dados tratados)
CREATE OR REPLACE TEMPORARY VIEW upserts
AS (
     SELECT 
          CdVen as CodigoVenda
        , DtVen as DataVenda
        , CdCli as CodigoCliente
        , NmCli as NomeCliente
        , Cidade
        , Pais
        , CdPro as CodigoProduto
        , NmPro as NomeProduto
        , TpPro as TipoProduto
        , cast(Qtd as int)             as Quantidade
        , Und
        , cast(VrUnt as decimal(18,2)) as ValorUnitario
        , CdVdd                        as CodigoVendedor
        , delete                       as RegistroExcluido
        , Estado
     FROM tbvendas_20210607
)

<h4> Analisando os schemas das tabelas aqui envolvidas </h4>

In [0]:
DESCRIBE silver_vendas;

col_name,data_type,comment
CodigoVenda,int,null
DataVenda,date,null
CodigoCliente,int,null
NomeCliente,string,null
Cidade,string,null
Estado,string,null
Pais,string,null
CodigoProduto,int,null
NomeProduto,string,null
TipoProduto,string,null


In [0]:
DESCRIBE upserts

col_name,data_type,comment
CodigoVenda,int,null
DataVenda,date,null
CodigoCliente,int,null
NomeCliente,string,null
Cidade,string,null
Pais,string,null
CodigoProduto,int,null
NomeProduto,string,null
TipoProduto,string,null
Quantidade,int,null


<h4> Verificando os registros que serão movimentados </h4>

In [0]:
-- Verificando...
SELECT * FROM upserts;

CodigoVenda,DataVenda,CodigoCliente,NomeCliente,Cidade,Pais,CodigoProduto,NomeProduto,TipoProduto,Quantidade,Und,ValorUnitario,CodigoVendedor,RegistroExcluido,Estado
2,2021-05-31,2,Cliente AB,Fortaleza,Brasil,2,Produto B,A,5000,KG,20.00,5,0,Ceara
10,2021-06-04,5,Cliente BB,Belo Horizonte,Brasil,1,Produto A,A,24000,KG,10.00,1,1,Minas Gerais
11,2021-06-07,3,Cliente BC,Baturite,Brasil,1,Produto A,A,5000,KG,10.00,2,0,Ceara
12,2021-06-07,4,Cliente CC,Fortaleza,Brasil,1,Produto A,A,6000,KG,10.00,1,0,Ceara
13,2021-06-07,5,Cliente BB,Belo Horizonte,Brasil,1,Produto A,A,8000,KG,10.00,3,0,Minas Gerais
14,2021-06-07,6,Cliente DD,Goiania,Brasil,1,Produto A,A,12500,KG,10.00,2,0,Goias
15,2021-06-07,7,Cliente DE,Joao Pessoa,Brasil,1,Produto A,A,12000,KG,10.00,2,0,Paraiba
16,2021-06-07,8,Cliente EE,Natal,Brasil,1,Produto A,A,17500,KG,10.00,3,0,Rio Grande do Norte
17,2021-06-07,5,Cliente BB,Belo Horizonte,Brasil,1,Produto A,A,24000,KG,10.00,1,0,Minas Gerais


<h4> Aplicando o MERGE </h4>

In [0]:
-- A palavra "upsert" é uma mala de viagem das palavras "atualizar" e "inserir", e é isso que ela faz. Um upsert irá atualizar os registros onde alguns critérios são atendidos e, caso contrário, irá inserir o registro. 
-- Ao fazer o upsert em uma tabela Delta existente, use o Spark SQL para realizar a mesclagem de outra tabela ou visualização registrada. O Log de transações registra a transação e o Metastore reflete imediatamente as alterações. 
-- A mesclagem anexa os arquivos novos / inseridos e os arquivos contendo as atualizações ao diretório de arquivos Delta. O log de transações informa ao leitor Delta qual arquivo usar para cada registro.

MERGE INTO silver_vendas                                   -- the MERGE instruction is used to perform the upsert
USING upserts

ON silver_vendas.CodigoVenda = upserts.CodigoVenda          -- AND silver_vendas.CodigoCliente = upserts.CodigoCliente  -- ON is used to describe the MERGE condition
   
WHEN MATCHED THEN                                           -- WHEN MATCHED describes the update behavior (pelo negócio, só estes campos podem ser alterados após a venda)
  UPDATE SET
  silver_vendas.Quantidade     = upserts.Quantidade,         
  silver_vendas.ValorUnitario  = upserts.ValorUnitario,
  silver_vendas.CodigoVendedor = upserts.CodigoVendedor,
  silver_vendas.RegistroExcluido = upserts.RegistroExcluido
WHEN NOT MATCHED THEN                                       -- WHEN NOT MATCHED describes the insert behavior
  INSERT (CodigoVenda, DataVenda, CodigoCliente, NomeCliente, Cidade, Pais, CodigoProduto, NomeProduto, TipoProduto, Quantidade, Und, ValorUnitario, CodigoVendedor, RegistroExcluido, Estado)              
  VALUES (CodigoVenda, DataVenda, CodigoCliente, NomeCliente, Cidade, Pais, CodigoProduto, NomeProduto, TipoProduto, Quantidade, Und, ValorUnitario, CodigoVendedor, RegistroExcluido, Estado)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
9,2,0,7


<h4> Viagem no Tempo </h4>
- Versão 0

In [0]:
-- Versão 0
SELECT * FROM silver_vendas VERSION AS OF 0

CodigoVenda,DataVenda,CodigoCliente,NomeCliente,Cidade,Estado,Pais,CodigoProduto,NomeProduto,TipoProduto,Quantidade,Und,ValorUnitario,CodigoVendedor,RegistroExcluido
3,2021-06-01,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,4200,KG,10.00,4,0
6,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,11200,KG,10.00,3,0
10,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,24000,KG,10.00,1,0
2,2021-05-31,2,Cliente AB,Fortaleza,Ceara,Brasil,2,Produto B,A,730,KG,20.00,5,0
4,2021-06-02,3,Cliente BC,Baturite,Ceara,Brasil,1,Produto A,A,250,KG,10.00,2,0
5,2021-06-03,4,Cliente CC,Fortaleza,Ceara,Brasil,1,Produto A,A,4500,KG,10.00,1,0
1,2021-01-20,1,Cliente AA,Florianopolis,Santa Catarina,Brasil,1,Produto A,A,4000,KG,8.00,2,0
9,2021-06-04,8,Cliente EE,Natal,Rio Grande do Norte,Brasil,1,Produto A,A,17500,KG,10.00,3,0
8,2021-06-04,7,Cliente DE,Joao Pessoa,Paraiba,Brasil,1,Produto A,A,12000,KG,10.00,2,0
7,2021-06-04,6,Cliente DD,Goiania,Goias,Brasil,1,Produto A,A,12500,KG,10.00,2,0


<h4> Viagem no Tempo </h4>
- Versão atual

In [0]:
SELECT *FROM silver_vendas;

CodigoVenda,DataVenda,CodigoCliente,NomeCliente,Cidade,Estado,Pais,CodigoProduto,NomeProduto,TipoProduto,Quantidade,Und,ValorUnitario,CodigoVendedor,RegistroExcluido
10,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,24000,KG,10.00,1,1
13,2021-06-07,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,8000,KG,10.00,3,0
17,2021-06-07,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,24000,KG,10.00,1,0
3,2021-06-01,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,4200,KG,10.00,4,0
6,2021-06-04,5,Cliente BB,Belo Horizonte,Minas Gerais,Brasil,1,Produto A,A,11200,KG,10.00,3,0
2,2021-05-31,2,Cliente AB,Fortaleza,Ceara,Brasil,2,Produto B,A,5000,KG,20.00,5,0
11,2021-06-07,3,Cliente BC,Baturite,Ceara,Brasil,1,Produto A,A,5000,KG,10.00,2,0
12,2021-06-07,4,Cliente CC,Fortaleza,Ceara,Brasil,1,Produto A,A,6000,KG,10.00,1,0
4,2021-06-02,3,Cliente BC,Baturite,Ceara,Brasil,1,Produto A,A,250,KG,10.00,2,0
5,2021-06-03,4,Cliente CC,Fortaleza,Ceara,Brasil,1,Produto A,A,4500,KG,10.00,1,0


In [0]:
select * from gold_fato_vendas;

CodigoVenda,DataVenda,Quantidade,ValorUnitario,CodigoCliente,CodigoProduto,CodigoVendedor
13,2021-06-07,8000,10.00,5,1,3
17,2021-06-07,24000,10.00,5,1,1
3,2021-06-01,4200,10.00,5,1,4
6,2021-06-04,11200,10.00,5,1,3
2,2021-05-31,5000,20.00,2,2,5
11,2021-06-07,5000,10.00,3,1,2
12,2021-06-07,6000,10.00,4,1,1
4,2021-06-02,250,10.00,3,1,2
5,2021-06-03,4500,10.00,4,1,1
16,2021-06-07,17500,10.00,8,1,3
